# NUMERIA-Q — teste de acesso ao Google Quantum AI

Notebook isolado para verificar credenciais e listar processadores autorizados, e então executar um teste físico de 1 qubit. Não usa nem modifica as versões congeladas 05.19.1 ou 05.20.

A execução real só ocorrerá na célula explicitamente marcada EXECUTAR NO HARDWARE. Sem acesso aprovado, o notebook para com uma mensagem clara; não faz fallback silencioso para simulação.

In [ ]:
%pip -q install cirq cirq-google

import cirq
import cirq_google as cg
print('Cirq:', cirq.__version__)
print('cirq_google importado com sucesso')

## 1. Informe o ID do projeto Google Cloud

Crie/selecione um projeto, habilite a Quantum Engine API e confirme que sua conta está autorizada. Não cole tokens ou senhas no notebook. O ID do projeto não é uma credencial secreta.

In [ ]:
PROJECT_ID = 'COLOQUE_AQUI_O_ID_DO_PROJETO'
if not PROJECT_ID or PROJECT_ID == 'COLOQUE_AQUI_O_ID_DO_PROJETO':
    raise ValueError('Edite PROJECT_ID com o ID do seu projeto Google Cloud antes de continuar.')

from google.colab import auth
auth.authenticate_user(clear_output=False)
print('Autenticação Google concluída. Isso, por si só, não confirma autorização ao hardware.')

In [ ]:
# Verificar acesso à API e listar os processadores visíveis à sua conta.
engine = cg.Engine(project_id=PROJECT_ID)
try:
    processors = list(engine.list_processors())
    available = []
    for p in processors:
        try:
            health = p.health()
        except Exception as exc:
            health = f'health indisponível: {type(exc).__name__}'
        available.append((p.processor_id, health))
    print('Processadores visíveis:', available)
    if not processors:
        print('Nenhum processador retornado. Acesso pode não estar aprovado ou não há processadores visíveis neste projeto.')
except Exception as exc:
    print('Não foi possível listar processadores.')
    print('Tipo do erro:', type(exc).__name__)
    print('Mensagem:', str(exc)[:1200])
    print('Verifique aprovação, Quantum Engine API, IAM e PROJECT_ID. Nenhum circuito físico foi enviado.')
    raise

## 2. EXECUTAR NO HARDWARE — teste mínimo de 1 qubit

Só execute depois de confirmar que a lista acima contém um processador autorizado em estado utilizável. O teste aplica X e mede; o resultado ideal é sempre 1. Usamos 20 repetições. Ruído pode produzir alguns zeros. Esta célula envia um trabalho ao serviço real.

In [ ]:
PROCESSOR_ID = 'COLOQUE_AQUI_O_PROCESSOR_ID'
REPETITIONS = 20
if not PROCESSOR_ID or PROCESSOR_ID == 'COLOQUE_AQUI_O_PROCESSOR_ID':
    raise ValueError('Preencha PROCESSOR_ID com um ID retornado por engine.list_processors().')
if not any(pid == PROCESSOR_ID for pid, _ in available):
    raise ValueError('PROCESSOR_ID não consta na lista visível. Não enviar circuito.')

processor = engine.get_processor(PROCESSOR_ID)
device = processor.get_device()
qubits = sorted(device.metadata.qubit_set)
if not qubits:
    raise RuntimeError('O dispositivo não expôs qubits utilizáveis.')
q = qubits[0]
circuit = cirq.Circuit(cirq.X(q), cirq.measure(q, key='m'))
device.validate_circuit(circuit)
print('Processador:', PROCESSOR_ID)
print('Qubit:', q)
print(circuit)
sampler = processor.get_sampler()
result = sampler.run(circuit, repetitions=REPETITIONS)
hist = result.histogram(key='m')
ones = hist.get(1, 0)
zeros = hist.get(0, 0)
print('Histograma:', hist)
print(f'1s={ones}; 0s={zeros}; total={ones+zeros}')
print('TRABALHO FÍSICO CONCLUÍDO: resultado retornado pelo sampler do processador Google.')
print('Critério básico: resultado recebido e 20 amostras. Ideal sem ruído: 20 resultados iguais a 1.')

## 3. Registro de auditoria

Copie para o diário NUMERIA-Q: data/hora, ID do projeto, ID do processador, qubit, repetições, histograma, ID/URL do job se mostrado no console, versão de Cirq e eventuais erros. Não registre tokens nem credenciais.

Este teste valida somente acesso e execução de um circuito de 1 qubit; não valida ainda os algoritmos reversíveis NUMERIA-Q nem reproduz resultados da Anthropic.